# CardioShift — Know When Not to Predict

## 1. Executive Summary

This executable report evaluates transportability across four historical
hospital cohorts. Headline values below are emitted by the shared results
accessor from validated run artifacts; they are not copied into prose.

In [ ]:
from pathlib import Path
import hashlib, json, os, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def locate_data() -> Path:
    configured = os.environ.get("CARDIOSHIFT_DATA_DIR")
    candidates = []
    if configured:
        candidates.append(Path(configured))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd / "dist" / "kaggle" / "cardioshift-data"])
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend(path.parent for path in kaggle_root.rglob("manifest.json"))
    for candidate in candidates:
        if (candidate / "outputs" / "results.json").exists():
            return candidate
    raise FileNotFoundError("Set CARDIOSHIFT_DATA_DIR to the Kaggle Dataset directory")

BASE = locate_data()
SOURCE = BASE / "source"
if SOURCE.exists():
    sys.path.insert(0, str(SOURCE))
elif str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
from src.results_access import ResultsAccessor

OUTPUT_DIR = (
    Path("/kaggle/working/cardioshift_outputs")
    if Path("/kaggle/working").exists()
    else Path(os.environ.get("CARDIOSHIFT_OUTPUT_DIR", Path.cwd() / "cardioshift_outputs"))
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
R = ResultsAccessor(BASE)
F = R.findings
print({
    "random_split_mean_auroc": round(F["random_split_mean_auroc"], 3),
    "loho_pooled_auroc": round(F["loho_pooled_auroc"], 3),
    "optimism_gap": round(F["random_minus_loho_auroc"], 3),
    "safety_coverage": round(F["safety_gate_coverage"], 3),
    "selective_error": round(F["safety_gate_selective_error"], 3),
})

## 2. Intended Use and Non-use

This is a retrospective research benchmark for studying hospital shift,
calibration, selective prediction, and failure modes. It does not predict
future risk and is not for diagnosis, treatment, medication, triage, or
real-patient decisions.

## 3. Dataset Provenance and License

The four processed center files come from the UCI Heart Disease dataset
(DOI 10.24432/C52P4X), licensed CC BY 4.0. The offline bundle includes the
source files, checksums, attribution, source archive, and reproducibility
manifest.

In [ ]:
manifest_path = BASE / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    failures = []
    platform_transforms = []
    for relative, expected in manifest["files"].items():
        path = BASE / relative
        if not path.is_file():
            unpacked = path.with_suffix("")
            if Path("/kaggle/input").exists() and path.suffix.lower() == ".zip" and unpacked.is_dir():
                platform_transforms.append(relative)
                continue
            failures.append(f"{relative}: missing")
            continue
        content = path.read_bytes()
        if path.suffix.lower() in {".json", ".csv", ".md", ".py", ".yaml", ".yml"}:
            content = content.replace(b"\r\r\n", b"\n").replace(b"\r\n", b"\n")
        actual = hashlib.sha256(content).hexdigest()
        if actual != expected["sha256"]:
            failures.append(relative)
    assert not failures, failures
    print(f"Offline manifest verified: {len(manifest['files'])} files")
    print(f"Kaggle archive transforms: {platform_transforms}")
else:
    print("Local repository mode: canonical artifact hashes are verified by the test suite.")

## 4. Four-hospital cohort

The pooled cohort contains one deidentified row per record from Cleveland,
Hungary, Switzerland, and VA Long Beach. Missing values remain missing until
training-fold preprocessing.

In [ ]:
hospital = R.per_hospital()
display(hospital)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(hospital["hospital"], hospital["n"], color="#3b6ea8")
ax.set_ylabel("Records")
ax.set_title("Four-hospital cohort")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cohort_by_hospital.png", dpi=140)
plt.show()

## 5. Validation design

The primary design is leave-one-hospital-out (LOHO): every patient in the test
hospital is excluded from tuning, calibration, and final fitting. Repeated
random splitting is retained only as a contrast. Random splitting estimates
within-mixture performance; it is not deployment validation for a new hospital.

## 6. Random Split vs LOHO

In [ ]:
comparison = pd.DataFrame({
    "design": ["Repeated random split", "Leave-one-hospital-out"],
    "AUROC": [F["random_split_mean_auroc"], F["loho_pooled_auroc"]],
    "Brier": [F["random_split_mean_brier"], F["loho_pooled_brier"]],
})
display(comparison)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].bar(comparison["design"], comparison["AUROC"], color=["#579c87", "#d97757"])
axes[1].bar(comparison["design"], comparison["Brier"], color=["#579c87", "#d97757"])
axes[0].set_ylim(0.5, 1.0); axes[0].set_title("AUROC (higher is better)")
axes[1].set_ylim(0.0, 0.3); axes[1].set_title("Brier (lower is better)")
for ax in axes: ax.tick_params(axis="x", rotation=18)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "random_vs_loho.png", dpi=140)
plt.show()

## 7. Per-hospital calibration

In [ ]:
display(hospital)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(hospital["hospital"], hospital["auroc"], color="#d97757")
ax.axhline(F["loho_pooled_auroc"], color="black", linestyle="--", label="pooled")
ax.set_ylim(0.45, 1.0); ax.set_ylabel("AUROC"); ax.legend()
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "loho_by_hospital.png", dpi=140)
plt.show()

## 8. Dataset-shift diagnostics

Hospital identity is predictable from inputs, which is evidence of dataset
shift—not a clinical prediction task. Prevalence, missingness, and standardized
feature differences vary substantially by center.

In [ ]:
shift = R.canonical["experiments"]["E3_E5"]["E3_shift"]
print({
    "site_classifier_balanced_accuracy": F["site_classifier_balanced_accuracy"],
    "balanced_chance": F["site_classifier_chance"],
})
missing = pd.DataFrame(shift["missingness_by_site"]).T
display(missing)

## 9. Selective prediction

The prespecified gate combines an outer-training OOD threshold, a fixed
probability ambiguity band, class-conditional conformal sets, and a fixed
training missingness quantile. Deferral is a coverage–error tradeoff, not proof
of clinical safety.

In [ ]:
print({
    "coverage": F["safety_gate_coverage"],
    "deferral_rate": 1 - F["safety_gate_coverage"],
    "selective_error": F["safety_gate_selective_error"],
    "accepted_case_fnr": F["accepted_case_fnr"],
    "empirical_conformal_coverage": F["empirical_conformal_coverage"],
})

## 10. E6 robustness

In [ ]:
robust = R.robustness_summary()
display(robust)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(robust["scenario"], robust["auroc"], marker="o", label="AUROC")
ax.plot(robust["scenario"], robust["safety_coverage"], marker="s", label="Safety coverage")
ax.set_ylim(0, 1); ax.tick_params(axis="x", rotation=30); ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "robustness.png", dpi=140)
plt.show()

## 11. Subgroups

Sex and age-band analyses use accepted patient-level LOHO predictions. They are
descriptive only and do not establish fairness.

In [ ]:
display(R.subgroup_summary())

## 12. Runtime profile

The values below are local CPU research-runtime measurements, not
medical-device benchmarks.

In [ ]:
runtime = pd.DataFrame(R.runtime["by_fold"]).T
display(runtime[["serialized_model_bytes", "batch_1_ms", "batch_100_ms", "peak_process_memory_bytes"]])
print(R.runtime["environment"])

## 13. Failure case

VA Long Beach achieved the worst empirical conformal coverage. This matters
because the safety method itself can fail under hospital shift; the gate must
not be presented as a guarantee.

In [ ]:
worst = F["worst_site_conformal_coverage"]
print({"hospital": worst["site"], "empirical_conformal_coverage": worst["value"]})

## 14. Limitations

In [ ]:
for limitation in R.limitations:
    print(f"- {limitation}")

## 15. Reproducibility manifest

In [ ]:
summary = {
    "canonical_results_sha256": R.sha256("outputs/results.json"),
    "input_sha256": R.canonical["data"]["input_sha256"],
    "gate_status": R.gates,
    "output_directory": str(OUTPUT_DIR),
    "internet_used": False,
}
print(json.dumps(summary, indent=2))
(OUTPUT_DIR / "notebook_run_summary.json").write_text(json.dumps(summary, indent=2) + "\n")